## Feature Engineering

In this section we add to the dataset which we currently don't have

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from pathlib import Path
import sys
import os


path = os.getcwd()
while ".git" not in os.listdir(path):
    path = os.path.dirname(path)
ROOT_DIR = Path(path)
os.chdir(ROOT_DIR)

from scripts.helpers.datasets import load_taxi_data

In [ ]:
df = load_taxi_data(preprocessed=True, feature_Engineering=False)

In [ ]:
df['speed_mph'] = df['trip_miles'] / (df['trip_seconds'] / 3600)
df['speed_mph'].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).T.round(2)

count    5891622.00
mean          14.05
std            9.41
min            2.00
1%             3.45
5%             5.20
50%           10.32
95%           34.80
99%           45.20
99.9%         52.97
max           69.16
Name: speed_mph, dtype: float64

### Temporal Features

Ride demand is strongly driven by *when* a trip starts, so we derive calendar features from `trip_start_timestamp` that the later demand models (forecasting trips per spatial unit and time bucket) rely on. The timestamp is already timezone-aware (`America/Chicago`), so every field below is in local Chicago time.

Alongside the basic calendar fields (`hour`, `day_of_week`, `month`, …) we add a US (Illinois) public-holiday flag, since demand on holidays such as Independence Day or Christmas deviates strongly from a normal weekday. Cyclical (sin/cos) encodings — which stop the models from reading hour 23 and hour 0 as far apart — are intentionally **left for the modeling step**, where the encoding can be chosen per model.

All of these features are encapsulated in `preprocess_taxi_data()`, so any notebook calling `load_taxi_data(preprocessed=True)` receives them automatically.

In [ ]:
import holidays

# Calendar features derived from the (tz-aware, local) trip start time.
ts = df["trip_start_timestamp"].dt

df["date"]        = ts.date                              # calendar day (for daily aggregation)
df["hour"]        = ts.hour.astype("Int64")                              # 0–23
df["day_of_week"] = ts.dayofweek.astype("Int64")                         # 0 = Monday … 6 = Sunday
df["is_weekend"]  = ts.dayofweek.isin([5, 6])            # Sat/Sun flag
df["week"]        = ts.isocalendar().week.astype("Int64")  # ISO week number
df["month"]       = ts.month.astype("Int64")                             # 1–12
  
# US (Illinois) public-holiday flag(demand differs strongly on holidays)
years = range(int(ts.year.min()), int(ts.year.max()) + 1)
us_il_holidays = holidays.US(subdiv="IL", years=years)
df["is_holiday"] = df["date"].isin(set(us_il_holidays))

df[["trip_start_timestamp", "date", "hour", "day_of_week",
    "is_weekend", "week", "month", "is_holiday"]].head()

,trip_start_timestamp,date,hour,day_of_week,is_weekend,week,month,is_holiday
0,2024-01-01 00:00:00-06:00,2024-01-01,0,0,False,1,1,True
1,2024-01-01 00:00:00-06:00,2024-01-01,0,0,False,1,1,True
2,2024-01-01 00:00:00-06:00,2024-01-01,0,0,False,1,1,True
3,2024-01-01 00:00:00-06:00,2024-01-01,0,0,False,1,1,True
4,2024-01-01 00:15:00-06:00,2024-01-01,0,0,False,1,1,True


### Create cyclic Features

We create cyclic features from time-related columns (hour, day of week, month, year) so the model learns their periodic nature instead of treating them as ordinary integers.

**Why cyclic encoding?**

Many calendar variables “wrap around”: 23:00 is close to 00:00, Sunday is close to Monday, and December is close to January. If we keep these as integers, a regression model may interpret the step from 23 → 0 (or 12 → 1) as a large discontinuity, even though the points are neighbors in time. Cyclic encoding fixes this by mapping each value onto a point on the unit circle using sine and cosine, making nearby times have nearby representations.

**How it works (sin & cos pair)**

For a cyclic variable x with period P, we compute:


$$ x_{\sin}=\sin\left(2\pi\frac{x}{P}\right)$$
$$x_{\cos}=\cos\left(2\pi\frac{x}{P}\right)$$



Using both sine and cosine is important: sine alone cannot uniquely represent positions around a circle, while the pair (sin⁡,cos⁡)(sin,cos) preserves the full circular position.


In [ ]:
def create_cyclic_features(df):
    df = df.copy()

    # Hour of day
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # Day of week
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['dayofweek'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['dayofweek'] / 7)
    
    # Month of year
    df['month_sin'] = np.sin(2 * np.pi * (df['month'] - 1) / 12)
    df['month_cos'] = np.cos(2 * np.pi * (df['month'] - 1) / 12)
    
    return df


In [ ]:
df = create_cyclic_features(df)
df.head()